In [8]:
import os
os.environ["GLOG_minloglevel"] = "2"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

from pathlib import Path

import numpy as np
import ipywidgets as widgets
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
import threading

from temgym_core.components import (
    Lens, Detector, Plane, SigmoidAperture, DoubleDeflector,
)
from temgym_core.ray import Ray
from temgym_core.gaussian import make_gaussian
from temgym_core.run import run_to_end, run_to_end_vmapped, run_iter_vmapped
from temgym_core.evaluate import evaluate_gaussians_gpu_kernel_wrapper
from temgym_core.plotting import plot_model_plotly
from temgym_core.microscope_model import MicroscopeModel
from temgym_core.utils import fibonacci_spiral

import jax
import jax.numpy as jnp
import optimistix as optx

jax.config.update("jax_enable_x64", True)


In [9]:
model_path = Path("data/microscope_design_curves.npz")
model = MicroscopeModel.from_npz(model_path)

aux = dict(getattr(model, 'auxiliary', {}) or {})
source_waist_m = float(aux.get('source_waist_m', 0.5e-9))
z_source = float(aux.get('z_source', 0.0))

spot_values = np.asarray(model.modes['spot'].control_values, dtype=float)
mag_values = np.asarray(model.modes['mag'].control_values, dtype=float)

print(f"Loaded MicroscopeModel at {model.voltage/1000:.0f} kV")
print(f"Modes: {list(model.modes.keys())}")
print(f"Source waist: {source_waist_m*1e9:.2f} nm at z = {z_source:.4f} m")


Loaded MicroscopeModel at 200 kV
Modes: ['spot', 'mag']
Source waist: 10.00 nm at z = 0.0000 m


In [10]:
# ── Helper functions ──────────────────────────────────────────────────

GRID_PIXELS = int(aux.get('gui_grid_pixels', 256))
N_SOURCE_RAYS = 1_000_000
FAN_NUM_RAYS = 31
APERTURE_SHARPNESS = 1e14
APERTURE_RADII_UM = [5, 10, 15, 20, 30, 50, 75, 100, 150]
# Fixed source half-angle large enough to overfill any aperture in the list
SOURCE_THETA_RAD = 5e-3  # 5 mrad

# Deflector defaults from auxiliary dict
DEFLECTOR_SPACING = float(aux.get('deflector_spacing', 0.005))
_def_range = aux.get('deflector_drive_range', [-0.1, 0.1])
DEFLECTOR_DRIVE_RANGE = [min(_def_range[0], -0.1), max(_def_range[1], 0.1)]
DEFLECTOR_BALANCE_RANGE = aux.get('deflector_balance_range', [0.2, 3.0])


def make_point_source(n, theta_max, waist, voltage, z0=0.0):
    """Point source at (0,0) with E(0,0) = 1, flat wavefront, pathlength = 0."""
    n = int(n)
    dx, dy = fibonacci_spiral(n, float(theta_max))
    dx = np.asarray(dx, dtype=float)
    dy = np.asarray(dy, dtype=float)
    zeros = np.zeros(n, dtype=float)
    return make_gaussian(
        x=zeros, y=zeros, dx=dx, dy=dy,
        z=np.full(n, float(z0), dtype=float),
        voltage=np.full(n, float(voltage), dtype=float),
        waist_x=np.full(n, float(waist), dtype=float),
        waist_y=np.full(n, float(waist), dtype=float),
        amp=np.full(n, 1.0 / max(n, 1), dtype=float),
        phase=zeros,
        rcurv_x=np.full(n, np.inf, dtype=float),
        rcurv_y=np.full(n, np.inf, dtype=float),
        wavelength_unit='m',
    )


def em_to_lens(em):
    return Lens(z=float(em.z), focal_length=float(em.focal_length))


def make_detector(z, half_width, shape=GRID_PIXELS):
    pixel = (2.0 * float(half_width)) / float(shape)
    return Detector(z=float(z), pixel_size=(pixel, pixel), shape=(int(shape), int(shape)))


def intensity_image(beam, detector):
    field = np.asarray(evaluate_gaussians_gpu_kernel_wrapper(beam, detector))
    return np.abs(field) ** 2


def _abcd_free(d):
    return np.array([[1.0, float(d)], [0.0, 1.0]])


def _abcd_lens(f):
    return np.array([[1.0, 0.0], [-1.0 / float(f), 1.0]])


def abcd_source_to_z(components, z_target, z_src=0.0):
    """Build ABCD matrix from z_src through components up to z_target."""
    M = np.eye(2)
    z_prev = float(z_src)
    for c in components:
        z_c = float(getattr(c, 'z'))
        if z_c > z_target + 1e-12:
            break
        M = _abcd_free(z_c - z_prev) @ M
        z_prev = z_c
        if isinstance(c, Lens):
            M = _abcd_lens(c.focal_length) @ M
    # final drift to z_target
    if z_prev < z_target - 1e-12:
        M = _abcd_free(z_target - z_prev) @ M
    return M


def beam_extent_at_z(components, theta_max, z_target, z_src=0.0):
    """Estimate beam half-width at z_target from ABCD.

    Returns |B| * theta_max (the extent of outermost ray from on-axis source).
    """
    M = abcd_source_to_z(components, z_target, z_src)
    return abs(M[0, 1]) * float(theta_max)


def build_microscope(spot_val, mag_val, aperture_radius_um=30.0,
                     aperture_x_um=0.0, aperture_y_um=0.0,
                     deflector_shift_x=0.0, deflector_shift_y=0.0,
                     deflector_tilt_x=0.0, deflector_tilt_y=0.0,
                     deflector_shift_balance_x=1.0, deflector_shift_balance_y=1.0,
                     deflector_tilt_balance_x=1.0, deflector_tilt_balance_y=1.0):
    """Build the full component list and metadata for a given spot/mag setting."""
    spot_lenses = model.build_components('spot', float(spot_val))
    mag_lenses = model.build_components('mag', float(mag_val))

    CL1_em, CL3_em, C_mini_em, Obj_prefield_em = spot_lenses[:4]
    Obj_post_em, IL1_em, IL2_em, IL3_em, PL1_em = mag_lenses[4:]

    z_sample = float(aux.get('z_sample',
                             Obj_prefield_em.z + abs(Obj_prefield_em.focal_length)))
    d_pl1_to_det = float(aux.get('d_pl1_to_detector_m', 0.325))
    z_detector = float(aux.get('z_detector', PL1_em.z + d_pl1_to_det))

    z_aperture = float(aux.get('z_aperture',
                               0.5 * (CL3_em.z + C_mini_em.z)))
    aperture = SigmoidAperture(
        z=z_aperture,
        radius=float(aperture_radius_um) * 1e-6,
        x0=float(aperture_x_um) * 1e-6,
        y0=float(aperture_y_um) * 1e-6,
        edge_width=1e-12,
        sharpness=APERTURE_SHARPNESS,
        t_low=0.0, t_high=1.0,
    )

    # DoubleDeflector placed between aperture and C_mini
    z_deflector = z_aperture + 0.5 * (float(C_mini_em.z) - z_aperture - DEFLECTOR_SPACING)
    deflector = DoubleDeflector(
        z=z_deflector,
        spacing=DEFLECTOR_SPACING,
        shift_x=float(deflector_shift_x),
        shift_y=float(deflector_shift_y),
        tilt_x=float(deflector_tilt_x),
        tilt_y=float(deflector_tilt_y),
        shift_balance_x=float(deflector_shift_balance_x),
        shift_balance_y=float(deflector_shift_balance_y),
        tilt_balance_x=float(deflector_tilt_balance_x),
        tilt_balance_y=float(deflector_tilt_balance_y),
    )

    components = [
        em_to_lens(CL1_em),
        em_to_lens(CL3_em),
        aperture,
        deflector,
        em_to_lens(C_mini_em),
        em_to_lens(Obj_prefield_em),
        Plane(z=z_sample),
        em_to_lens(Obj_post_em),
        em_to_lens(IL1_em),
        em_to_lens(IL2_em),
        em_to_lens(IL3_em),
        em_to_lens(PL1_em),
        Plane(z=z_detector),
    ]

    labels = [
        "CL1", "CL3", "Aperture", "Deflector", "C_mini", "Obj_pre", "Sample",
        "Obj_post", "IL1", "IL2", "IL3", "PL1", "Detector",
    ]

    plot_components = [
        CL1_em, CL3_em, Plane(z=z_aperture),
        deflector,
        C_mini_em, Obj_prefield_em,
        Plane(z=z_sample),
        Obj_post_em, IL1_em, IL2_em, IL3_em, PL1_em,
        make_detector(z_detector, 15e-3, shape=GRID_PIXELS),
    ]

    return {
        'components': components,
        'plot_components': plot_components,
        'labels': labels,
        'z_sample': z_sample,
        'z_detector': z_detector,
        'z_aperture': z_aperture,
        'z_deflector': z_deflector,
        'aperture_radius_m': float(aperture_radius_um) * 1e-6,
        'aperture_x_um': float(aperture_x_um),
        'aperture_y_um': float(aperture_y_um),
    }


def make_source_beam(n_rays=N_SOURCE_RAYS):
    """Build point-source beam with the fixed source half-angle."""
    return make_point_source(
        n=int(n_rays),
        theta_max=SOURCE_THETA_RAD,
        waist=source_waist_m,
        voltage=model.voltage,
        z0=z_source,
    )


def propagate_steps(beam, components):
    """Return [input, after_comp0, after_comp1, ...].

    ``run_iter_vmapped`` yields two outputs per component (propagator
    then component).  We keep only the component outputs (odd indices)
    so ``beam_steps[i+1]`` is the state after ``components[i]``.
    """
    all_outputs = run_iter_vmapped(beam, components)
    steps = [beam]
    steps.extend(all_outputs[1::2])  # skip propagator intermediates
    return steps


def beam_image_at_z(beam_steps, components, z_target, half_width):
    """Evaluate intensity at an arbitrary z plane using the nearest upstream state."""
    comp_zs = [float(getattr(c, 'z')) for c in components]
    best_idx = 0
    for i, cz in enumerate(comp_zs):
        if cz <= z_target + 1e-12:
            best_idx = i + 1
    beam_state = beam_steps[min(best_idx, len(beam_steps) - 1)]
    beam_z = float(np.asarray(beam_state.z).ravel()[0])
    if abs(beam_z - z_target) > 1e-12:
        beam_state = run_to_end_vmapped(beam_state, [Plane(z=z_target)])
    det = make_detector(z_target, half_width, shape=GRID_PIXELS)
    return intensity_image(beam_state, det), det


def make_ray_fan(theta0):
    dx_fan = np.linspace(-float(theta0), float(theta0), FAN_NUM_RAYS)
    return Ray(
        x=np.zeros_like(dx_fan), y=np.zeros_like(dx_fan),
        dx=dx_fan, dy=np.zeros_like(dx_fan),
        z=np.full_like(dx_fan, z_source),
        pathlength=np.zeros_like(dx_fan),
    )


# ── Solver helpers for double deflector balance ──────────────────────

def pivot_distance(spacing_val, balance_val, eps=1e-12):
    """Distance from second deflector to pivot point."""
    if abs(float(balance_val) - 1.0) < eps:
        return np.inf
    return float(spacing_val) / (float(balance_val) - 1.0)


def bounded_logistic_to_physical(u, bounds_arr):
    b = jnp.asarray(bounds_arr, dtype=jnp.float64)
    lo, hi = b[:, 0], b[:, 1]
    return lo + (hi - lo) * jax.nn.sigmoid(u)


def bounded_logit_from_physical(x, bounds_arr):
    b = np.asarray(bounds_arr, dtype=float)
    lo, hi = b[:, 0], b[:, 1]
    s = np.clip((np.asarray(x, dtype=float) - lo) / (hi - lo), 1e-9, 1.0 - 1e-9)
    return np.log(s) - np.log1p(-s)


print("Helpers defined.")


Helpers defined.


In [11]:
# ── Interactive microscope viewer ──────────────────────────────────────

# Fundamental (waist/divergence) solution with the same theta extent as fan rays
source_basis = Ray(
    x=np.array([0.0, 0.0]),
    y=np.array([0.0, 0.0]),
    dx=np.array([0.0, SOURCE_THETA_RAD]),
    dy=np.array([0.0, 0.0]),
    z=np.array([z_source, z_source]),
    pathlength=np.array([0.0, 0.0]),
)

RAY_COUNT_OPTIONS = [1_000, 10_000, 100_000, 1_000_000, 10_000_000]
Z_STEP_OPTIONS_UM = [0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0]
APERTURE_STEP_OPTIONS_UM = [1.0, 5.0, 10.0, 50.0, 100.0]


def _format_um_step(step_um):
    if step_um < 0.1:
        return f"{int(round(step_um * 1000.0))} nm"
    if float(step_um).is_integer():
        return f"{int(step_um)} um"
    return f"{step_um:g} um"


# Initial build
init_spot = float(spot_values[0])
init_mag = float(mag_values[0])
init_ap_um = 30.0
init_ray_count = N_SOURCE_RAYS
micro = build_microscope(init_spot, init_mag, aperture_radius_um=init_ap_um)

beam_in = make_source_beam(init_ray_count)
beam_steps = propagate_steps(beam_in, micro['components'])
fan_rays = make_ray_fan(SOURCE_THETA_RAD)

comp_zs = [float(getattr(c, 'z')) for c in micro['components']]
all_zs = sorted(set([z_source] + comp_zs))
z_min, z_max = float(min(all_zs)), float(max(all_zs))

# Start at source
init_z = z_source
init_hw = beam_extent_at_z(micro['components'], SOURCE_THETA_RAD, init_z, z_source)
init_hw = max(init_hw, source_waist_m * 10) * 1.5
init_image, init_det = beam_image_at_z(beam_steps, micro['components'], init_z, init_hw)
init_extent = np.asarray(init_det.extent) * 1e6  # um

# ── Ray diagram ──────────────────────────────────────────────────────
ray_fig = plot_model_plotly(
    micro['plot_components'],
    rays=fan_rays,
    solution_rays=source_basis,
    component_labels=micro['labels'],
    include_input_rays=True,
    band_mode='lines',
    ray_coordinate='x_rot',
    show_component_labels=False,
    width=600, height=700,
)

# ── Combined figure ──────────────────────────────────────────────────
fig = go.FigureWidget(
    make_subplots(
        rows=1, cols=2,
        column_widths=[0.40, 0.60],
        horizontal_spacing=0.14,
        subplot_titles=("Beam intensity", "Ray diagram"),
    )
)


def _extent_params(extent_um, shape):
    """Return center-origin + spacing matching matplotlib extent semantics."""
    x0, x1, y0, y1 = [float(v) for v in extent_um]
    ny, nx = int(shape[0]), int(shape[1])
    dx = (x1 - x0) / max(nx, 1)
    dy = (y1 - y0) / max(ny, 1)
    return x0 + 0.5 * dx, dx, y0 + 0.5 * dy, dy


x0_init, dx_init, y0_init, dy_init = _extent_params(init_extent, init_image.shape)

# Heatmap with non-overlapping colorbar
fig.add_trace(
    go.Heatmap(
        z=init_image.tolist(),
        x0=float(x0_init),
        dx=float(dx_init),
        y0=float(y0_init),
        dy=float(dy_init),
        zsmooth=False,
        colorscale='Inferno',
        zmin=0.0,
        zmax=max(float(np.max(init_image)), 1e-30),
        showscale=True,
        colorbar=dict(
            len=1.0,
            y=0.5,
            yanchor='middle',
            x=0.34,
            xanchor='left',
            thickness=10,
            title=dict(text='I', side='right'),
        ),
    ),
    row=1, col=1,
 )
IMAGE_TRACE_IDX = 0

# Aperture circle overlay on heatmap (hidden initially, shown near aperture z)
theta_circle = np.linspace(0, 2 * np.pi, 200)
fig.add_trace(
    go.Scatter(
        x=(micro['aperture_radius_m'] * np.cos(theta_circle) * 1e6).tolist(),
        y=(micro['aperture_radius_m'] * np.sin(theta_circle) * 1e6).tolist(),
        mode='lines',
        line=dict(color='cyan', width=2, dash='dash'),
        showlegend=False,
        hoverinfo='skip',
        visible=False,
    ),
    row=1, col=1,
 )
APERTURE_CIRCLE_IDX = 1

# Ray traces (col 2)
RAY_TRACE_START = 2
for trace in ray_fig.data:
    if getattr(trace, 'mode', None) == 'text':
        continue
    cloned = go.Figure(data=[trace]).data[0]
    fig.add_trace(cloned, row=1, col=2)
RAY_TRACE_END = len(fig.data)

# z-indicator line on ray diagram
fig.add_trace(
    go.Scatter(
        x=[-15e-3, 15e-3], y=[init_z, init_z],
        mode='lines',
        line=dict(color='red', width=2, dash='dash'),
        showlegend=False, hoverinfo='skip',
    ),
    row=1, col=2,
 )
Z_LINE_IDX = len(fig.data) - 1

# Aperture indicator on ray diagram: two horizontal lines with a gap
ap_r_plot = micro['aperture_radius_m']
max_x_ray = 15e-3  # plot half-width in metres
z_ap = micro['z_aperture']
fig.add_trace(
    go.Scatter(
        x=[-max_x_ray, -ap_r_plot], y=[z_ap, z_ap],
        mode='lines',
        line=dict(color='orange', width=4),
        showlegend=False, hoverinfo='skip',
    ),
    row=1, col=2,
 )
AP_RAY_LEFT_IDX = len(fig.data) - 1
fig.add_trace(
    go.Scatter(
        x=[ap_r_plot, max_x_ray], y=[z_ap, z_ap],
        mode='lines',
        line=dict(color='orange', width=4),
        showlegend=False, hoverinfo='skip',
    ),
    row=1, col=2,
 )
AP_RAY_RIGHT_IDX = len(fig.data) - 1

# Axes
fig.update_xaxes(title_text='x [um]', title_standoff=6, row=1, col=1)
fig.update_yaxes(
    title_text='y [um]',
    title_standoff=6,
    scaleanchor='x',
    scaleratio=1,
    constrain='domain',
    row=1, col=1,
)
fig.update_xaxes(title_text='x_rot [m]', range=[-max_x_ray, max_x_ray], row=1, col=2)
fig.update_yaxes(title_text='z [m]', autorange='reversed', row=1, col=2)
fig.update_xaxes(range=[float(init_extent[0]), float(init_extent[1])], row=1, col=1)
fig.update_yaxes(range=[float(init_extent[2]), float(init_extent[3])], row=1, col=1)

# Component labels on right side
x3_domain = fig.layout.xaxis2.domain
label_x = float(x3_domain[1]) + 0.01 if x3_domain else 0.68
label_annotations = []
for name, cz in zip(micro['labels'], comp_zs):
    label_annotations.append(dict(
        x=label_x, y=cz,
        xref='paper', yref='y2',
        text=f"{name} (z={cz:.4f})",
        showarrow=False, xanchor='left', font=dict(size=10),
    ))
base_annotations = list(fig.layout.annotations or [])
fig.layout.annotations = tuple(base_annotations + label_annotations)

fig.update_layout(
    height=700, width=1440,
    margin=dict(l=40, r=170, t=50, b=20),
    plot_bgcolor='white', paper_bgcolor='white',
    uirevision='keep',
)

def _sync_colorbar_to_heatmap():
    """Match colorbar height to the actually rendered heatmap height."""
    x_dom = fig.layout.xaxis.domain
    y_dom = fig.layout.yaxis.domain
    x_rng = fig.layout.xaxis.range
    y_rng = fig.layout.yaxis.range
    if not (x_dom and y_dom and x_rng and y_rng):
        return

    m = fig.layout.margin
    plot_w = float(fig.layout.width) - float(m.l) - float(m.r)
    plot_h = float(fig.layout.height) - float(m.t) - float(m.b)
    if plot_w <= 0 or plot_h <= 0:
        return

    x_dom_frac = float(x_dom[1]) - float(x_dom[0])
    y_dom_frac = float(y_dom[1]) - float(y_dom[0])
    x_span = abs(float(x_rng[1]) - float(x_rng[0]))
    y_span = abs(float(y_rng[1]) - float(y_rng[0]))
    if x_span <= 0 or y_span <= 0:
        return

    x_dom_px = x_dom_frac * plot_w
    y_dom_px = y_dom_frac * plot_h

    px_per_unit = x_dom_px / x_span
    y_needed_px = px_per_unit * y_span
    y_drawn_px = min(y_needed_px, y_dom_px)
    y_drawn_frac = y_drawn_px / plot_h

    y_center = 0.5 * (float(y_dom[0]) + float(y_dom[1]))
    x_right = float(x_dom[1])

    fig.data[IMAGE_TRACE_IDX].colorbar.len = float(max(y_drawn_frac, 1e-6))
    fig.data[IMAGE_TRACE_IDX].colorbar.y = float(y_center)
    fig.data[IMAGE_TRACE_IDX].colorbar.yanchor = 'middle'
    fig.data[IMAGE_TRACE_IDX].colorbar.x = float(x_right + 0.008)
    fig.data[IMAGE_TRACE_IDX].colorbar.xanchor = 'left'


_sync_colorbar_to_heatmap()

# ── Widgets ──────────────────────────────────────────────────────────

snap_options = [('-- free z --', None)]
for name, cz in zip(micro['labels'], comp_zs):
    snap_options.append((f"{name} (z={cz:.4f})", cz))
    if name == "Aperture":
        z_before_ap = cz - 1e-4  # 0.1 mm before
        snap_options.append((f"Before Aperture (z={z_before_ap:.4f})", z_before_ap))

spot_slider = widgets.SelectionSlider(
    options=[float(v) for v in spot_values],
    value=init_spot, description='Spot', continuous_update=False,
)
mag_slider = widgets.SelectionSlider(
    options=[float(v) for v in mag_values],
    value=init_mag, description='Mag', continuous_update=False,
)
ray_count_dropdown = widgets.Dropdown(
    options=[(f"{v:,}", int(v)) for v in RAY_COUNT_OPTIONS],
    value=int(init_ray_count), description='Beam rays',
    layout=widgets.Layout(width='180px'),
)
aperture_dropdown = widgets.Dropdown(
    options=[(f"{r} um", float(r)) for r in APERTURE_RADII_UM],
    value=init_ap_um, description='Radius',
    layout=widgets.Layout(width='170px'),
)
ap_step_dropdown = widgets.Dropdown(
    options=[(_format_um_step(step_um), float(step_um)) for step_um in APERTURE_STEP_OPTIONS_UM],
    value=5.0, description='Move step',
    layout=widgets.Layout(width='170px'),
)
z_slider = widgets.FloatSlider(
    value=init_z, min=z_min, max=z_max, step=0.0001,
    description='z [m]', continuous_update=False,
    readout_format='.4f',
    layout=widgets.Layout(width='600px'),
)
snap_dropdown = widgets.Dropdown(
    options=snap_options,
    value=None, description='Snap to:',
    layout=widgets.Layout(width='260px'),
)
hw_slider = widgets.FloatLogSlider(
    value=init_hw * 1e3, base=10,
    min=-7, max=2, step=0.1,
    description='HW [mm]', continuous_update=False,
    layout=widgets.Layout(width='260px'),
)
gain_slider = widgets.FloatLogSlider(
    value=1.0, base=10, min=-3, max=6, step=0.1,
    description='Gain', continuous_update=False,
    layout=widgets.Layout(width='230px'),
)
auto_extent_toggle = widgets.Checkbox(
    value=False, description='Auto extent',
    layout=widgets.Layout(width='130px'),
)

z_step_dropdown = widgets.Dropdown(
    options=[(_format_um_step(step_um), float(step_um)) for step_um in Z_STEP_OPTIONS_UM],
    value=1.0,
    description='dz',
    layout=widgets.Layout(width='150px'),
)
z_minus_btn = widgets.Button(description='-z', layout=widgets.Layout(width='44px'))
z_plus_btn = widgets.Button(description='+z', layout=widgets.Layout(width='44px'))

# Aperture position nudge buttons
ap_x_label = widgets.Label(value=f"x = {micro['aperture_x_um']:.1f} um", layout=widgets.Layout(width='90px'))
ap_y_label = widgets.Label(value=f"y = {micro['aperture_y_um']:.1f} um", layout=widgets.Layout(width='90px'))
ap_x_minus = widgets.Button(description='-x', layout=widgets.Layout(width='44px'))
ap_x_plus = widgets.Button(description='+x', layout=widgets.Layout(width='44px'))
ap_y_minus = widgets.Button(description='-y', layout=widgets.Layout(width='44px'))
ap_y_plus = widgets.Button(description='+y', layout=widgets.Layout(width='44px'))

# ── Deflector widgets ────────────────────────────────────────────────

_dr_lo, _dr_hi = float(DEFLECTOR_DRIVE_RANGE[0]), float(DEFLECTOR_DRIVE_RANGE[1])
_bl_lo, _bl_hi = float(DEFLECTOR_BALANCE_RANGE[0]), float(DEFLECTOR_BALANCE_RANGE[1])

shift_x_slider = widgets.FloatSlider(
    value=0.0, min=_dr_lo, max=_dr_hi, step=1e-4,
    description='Shift X', continuous_update=False,
    readout_format='.4f',
    layout=widgets.Layout(width='350px'),
)
shift_y_slider = widgets.FloatSlider(
    value=0.0, min=_dr_lo, max=_dr_hi, step=1e-4,
    description='Shift Y', continuous_update=False,
    readout_format='.4f',
    layout=widgets.Layout(width='350px'),
)
tilt_x_slider = widgets.FloatSlider(
    value=0.0, min=_dr_lo, max=_dr_hi, step=1e-4,
    description='Tilt X', continuous_update=False,
    readout_format='.4f',
    layout=widgets.Layout(width='350px'),
)
tilt_y_slider = widgets.FloatSlider(
    value=0.0, min=_dr_lo, max=_dr_hi, step=1e-4,
    description='Tilt Y', continuous_update=False,
    readout_format='.4f',
    layout=widgets.Layout(width='350px'),
)
shift_balance_x_slider = widgets.FloatSlider(
    value=1.0, min=_bl_lo, max=_bl_hi, step=0.01,
    description='Shift Bal X', continuous_update=False,
    readout_format='.3f',
    layout=widgets.Layout(width='350px'),
)
shift_balance_y_slider = widgets.FloatSlider(
    value=1.0, min=_bl_lo, max=_bl_hi, step=0.01,
    description='Shift Bal Y', continuous_update=False,
    readout_format='.3f',
    layout=widgets.Layout(width='350px'),
)
tilt_balance_x_slider = widgets.FloatSlider(
    value=1.0, min=_bl_lo, max=_bl_hi, step=0.01,
    description='Tilt Bal X', continuous_update=False,
    readout_format='.3f',
    layout=widgets.Layout(width='350px'),
)
tilt_balance_y_slider = widgets.FloatSlider(
    value=1.0, min=_bl_lo, max=_bl_hi, step=0.01,
    description='Tilt Bal Y', continuous_update=False,
    readout_format='.3f',
    layout=widgets.Layout(width='350px'),
)
pivot_shift_x_label = widgets.Label(
    value=f"Shift Pivot X: {pivot_distance(DEFLECTOR_SPACING, 1.0):.4f} m",
    layout=widgets.Layout(width='250px'),
)
pivot_shift_y_label = widgets.Label(
    value=f"Shift Pivot Y: {pivot_distance(DEFLECTOR_SPACING, 1.0):.4f} m",
    layout=widgets.Layout(width='250px'),
)
pivot_tilt_x_label = widgets.Label(
    value=f"Tilt Pivot X: {pivot_distance(DEFLECTOR_SPACING, 1.0):.4f} m",
    layout=widgets.Layout(width='250px'),
)
pivot_tilt_y_label = widgets.Label(
    value=f"Tilt Pivot Y: {pivot_distance(DEFLECTOR_SPACING, 1.0):.4f} m",
    layout=widgets.Layout(width='250px'),
)

# ── Wobble widgets ───────────────────────────────────────────────────

wobble_toggle = widgets.Checkbox(
    value=False, description='Wobble',
    layout=widgets.Layout(width='100px'),
)
wobble_amp_slider = widgets.FloatSlider(
    value=1e-3, min=1e-4, max=_dr_hi, step=1e-4,
    description='Ampl', continuous_update=False,
    readout_format='.4f',
    layout=widgets.Layout(width='280px'),
)
wobble_axis_dropdown = widgets.Dropdown(
    options=['x', 'y', 'both'],
    value='x', description='Axis',
    layout=widgets.Layout(width='130px'),
)
wobble_warning = widgets.Label(value='', layout=widgets.Layout(width='300px'))

# ── Solver widgets ───────────────────────────────────────────────────

solve_shift_balance_btn = widgets.Button(
    description='Solve Shift Bal',
    button_style='primary',
    layout=widgets.Layout(width='150px'),
)
solve_tilt_balance_btn = widgets.Button(
    description='Solve Tilt Bal',
    button_style='primary',
    layout=widgets.Layout(width='150px'),
)
solve_status_label = widgets.Label(value='', layout=widgets.Layout(width='500px'))

# ── Reset widget ─────────────────────────────────────────────────────

reset_deflector_btn = widgets.Button(
    description='Reset Deflector',
    button_style='warning',
    layout=widgets.Layout(width='140px'),
)

# ── State ────────────────────────────────────────────────────────────
_cache = {
    'micro': micro,
    'beam_steps': beam_steps,
    'beam_in': beam_in,
    'ap_x_um': float(micro['aperture_x_um']),
    'ap_y_um': float(micro['aperture_y_um']),
    'deflector_shift_x': 0.0,
    'deflector_shift_y': 0.0,
    'deflector_tilt_x': 0.0,
    'deflector_tilt_y': 0.0,
    'deflector_shift_balance_x': 1.0,
    'deflector_shift_balance_y': 1.0,
    'deflector_tilt_balance_x': 1.0,
    'deflector_tilt_balance_y': 1.0,
}

# Wobble state
_wobble_state = {
    'timer': None,
    'phase': 0.0,
}


def _update_snap_options(labels, comp_zs_new):
    opts = [('-- free z --', None)]
    for name, cz in zip(labels, comp_zs_new):
        opts.append((f"{name} (z={cz:.4f})", cz))
        if name == "Aperture":
            z_before = cz - 1e-4
            opts.append((f"Before Aperture (z={z_before:.4f})", z_before))
    snap_dropdown.options = opts


def _update_aperture_labels():
    ap_x_label.value = f"x = {_cache['ap_x_um']:.1f} um"
    ap_y_label.value = f"y = {_cache['ap_y_um']:.1f} um"


def _update_pivot_labels():
    sbx = _cache['deflector_shift_balance_x']
    sby = _cache['deflector_shift_balance_y']
    tbx = _cache['deflector_tilt_balance_x']
    tby = _cache['deflector_tilt_balance_y']
    psx = pivot_distance(DEFLECTOR_SPACING, sbx)
    psy = pivot_distance(DEFLECTOR_SPACING, sby)
    ptx = pivot_distance(DEFLECTOR_SPACING, tbx)
    pty = pivot_distance(DEFLECTOR_SPACING, tby)
    pivot_shift_x_label.value = (
        "Shift Pivot X: inf (parallel shift)" if np.isinf(psx)
        else f"Shift Pivot X: {psx*1e3:.2f} mm from def2"
    )
    pivot_shift_y_label.value = (
        "Shift Pivot Y: inf (parallel shift)" if np.isinf(psy)
        else f"Shift Pivot Y: {psy*1e3:.2f} mm from def2"
    )
    pivot_tilt_x_label.value = (
        "Tilt Pivot X: inf (parallel shift)" if np.isinf(ptx)
        else f"Tilt Pivot X: {ptx*1e3:.2f} mm from def2"
    )
    pivot_tilt_y_label.value = (
        "Tilt Pivot Y: inf (parallel shift)" if np.isinf(pty)
        else f"Tilt Pivot Y: {pty*1e3:.2f} mm from def2"
    )


def _aperture_step_um():
    return float(ap_step_dropdown.value)


def _update_aperture_on_ray_diagram(micro_dict):
    ap_r = micro_dict['aperture_radius_m']
    z_ap_local = micro_dict['z_aperture']
    fig.data[AP_RAY_LEFT_IDX].x = [-max_x_ray, -ap_r]
    fig.data[AP_RAY_LEFT_IDX].y = [z_ap_local, z_ap_local]
    fig.data[AP_RAY_RIGHT_IDX].x = [ap_r, max_x_ray]
    fig.data[AP_RAY_RIGHT_IDX].y = [z_ap_local, z_ap_local]


def rebuild_microscope(*_):
    _cache['deflector_shift_x'] = float(shift_x_slider.value)
    _cache['deflector_shift_y'] = float(shift_y_slider.value)
    _cache['deflector_tilt_x'] = float(tilt_x_slider.value)
    _cache['deflector_tilt_y'] = float(tilt_y_slider.value)
    _cache['deflector_shift_balance_x'] = float(shift_balance_x_slider.value)
    _cache['deflector_shift_balance_y'] = float(shift_balance_y_slider.value)
    _cache['deflector_tilt_balance_x'] = float(tilt_balance_x_slider.value)
    _cache['deflector_tilt_balance_y'] = float(tilt_balance_y_slider.value)

    micro_new = build_microscope(
        float(spot_slider.value),
        float(mag_slider.value),
        aperture_radius_um=float(aperture_dropdown.value),
        aperture_x_um=_cache['ap_x_um'],
        aperture_y_um=_cache['ap_y_um'],
        deflector_shift_x=_cache['deflector_shift_x'],
        deflector_shift_y=_cache['deflector_shift_y'],
        deflector_tilt_x=_cache['deflector_tilt_x'],
        deflector_tilt_y=_cache['deflector_tilt_y'],
        deflector_shift_balance_x=_cache['deflector_shift_balance_x'],
        deflector_shift_balance_y=_cache['deflector_shift_balance_y'],
        deflector_tilt_balance_x=_cache['deflector_tilt_balance_x'],
        deflector_tilt_balance_y=_cache['deflector_tilt_balance_y'],
    )
    beam_in_new = make_source_beam(int(ray_count_dropdown.value))
    beam_steps_new = propagate_steps(beam_in_new, micro_new['components'])
    _cache['micro'] = micro_new
    _cache['beam_steps'] = beam_steps_new
    _cache['beam_in'] = beam_in_new

    fan_new = make_ray_fan(SOURCE_THETA_RAD)
    new_ray_fig = plot_model_plotly(
        micro_new['plot_components'],
        rays=fan_new, solution_rays=source_basis,
        component_labels=micro_new['labels'],
        include_input_rays=True, band_mode='lines',
        ray_coordinate='x_rot', show_component_labels=False,
        width=600, height=700,
    )
    ray_traces = [t for t in new_ray_fig.data if getattr(t, 'mode', None) != 'text']

    new_comp_zs = [float(getattr(c, 'z')) for c in micro_new['components']]
    _update_snap_options(micro_new['labels'], new_comp_zs)

    new_labels = []
    for name, cz in zip(micro_new['labels'], new_comp_zs):
        new_labels.append(dict(
            x=label_x, y=cz,
            xref='paper', yref='y2',
            text=f"{name} (z={cz:.4f})",
            showarrow=False, xanchor='left', font=dict(size=10),
        ))

    with fig.batch_update():
        n_ray = min(RAY_TRACE_END - RAY_TRACE_START, len(ray_traces))
        for i in range(n_ray):
            src = ray_traces[i]
            fig.data[RAY_TRACE_START + i].x = list(src.x) if src.x is not None else []
            fig.data[RAY_TRACE_START + i].y = list(src.y) if src.y is not None else []
        fig.layout.annotations = tuple(base_annotations + new_labels)
        _update_aperture_on_ray_diagram(micro_new)

    _update_pivot_labels()
    update_beam_image()


def update_beam_image(*_):
    micro_now = _cache['micro']
    beam_steps_now = _cache['beam_steps']
    z_target = float(z_slider.value)
    gain = float(gain_slider.value)

    if auto_extent_toggle.value:
        hw_est = beam_extent_at_z(micro_now['components'], SOURCE_THETA_RAD, z_target, z_source)
        hw_est = max(hw_est, source_waist_m * 5) * 1.5
        hw = hw_est
        hw_slider.unobserve(update_beam_image, names='value')
        hw_slider.value = hw * 1e3
        hw_slider.observe(update_beam_image, names='value')
    else:
        hw = float(hw_slider.value) * 1e-3

    img, det = beam_image_at_z(beam_steps_now, micro_now['components'], z_target, hw)
    extent = np.asarray(det.extent) * 1e6  # um
    scaled = img * gain

    z_ap_local = micro_now['z_aperture']
    show_circle = abs(z_target - z_ap_local) < 2e-3
    ap_r_um = micro_now['aperture_radius_m'] * 1e6
    ap_x_um = _cache['ap_x_um']
    ap_y_um = _cache['ap_y_um']

    with fig.batch_update():
        x0_hm, dx_hm, y0_hm, dy_hm = _extent_params(extent, img.shape)
        fig.data[IMAGE_TRACE_IDX].z = scaled.tolist()
        fig.data[IMAGE_TRACE_IDX].x0 = float(x0_hm)
        fig.data[IMAGE_TRACE_IDX].dx = float(dx_hm)
        fig.data[IMAGE_TRACE_IDX].y0 = float(y0_hm)
        fig.data[IMAGE_TRACE_IDX].dy = float(dy_hm)
        fig.data[IMAGE_TRACE_IDX].zmax = max(float(np.max(scaled)), 1e-30)
        fig.layout.xaxis.range = [float(extent[0]), float(extent[1])]
        fig.layout.yaxis.range = [float(extent[2]), float(extent[3])]

        _sync_colorbar_to_heatmap()

        fig.data[Z_LINE_IDX].y = [z_target, z_target]

        fig.data[APERTURE_CIRCLE_IDX].visible = show_circle
        if show_circle:
            fig.data[APERTURE_CIRCLE_IDX].x = (ap_x_um + ap_r_um * np.cos(theta_circle)).tolist()
            fig.data[APERTURE_CIRCLE_IDX].y = (ap_y_um + ap_r_um * np.sin(theta_circle)).tolist()


def on_snap(change):
    val = change['new']
    if val is not None:
        z_slider.value = float(val)


def _nudge_z(direction):
    dz_m = float(z_step_dropdown.value) * 1e-6
    z_new = float(np.clip(float(z_slider.value) + direction * dz_m, float(z_slider.min), float(z_slider.max)))
    z_slider.value = z_new


def _nudge_aperture(dx_um, dy_um):
    step_um = _aperture_step_um()
    _cache['ap_x_um'] += dx_um * step_um
    _cache['ap_y_um'] += dy_um * step_um
    _update_aperture_labels()
    rebuild_microscope()


# ── Wobble logic ─────────────────────────────────────────────────────

def _wobble_tick():
    """One wobble tick: oscillate drive, rebuild, schedule next."""
    if not wobble_toggle.value:
        return
    amp = float(wobble_amp_slider.value)
    axis = wobble_axis_dropdown.value
    _wobble_state['phase'] += 0.15  # ~15% of a cycle per tick
    phase = _wobble_state['phase']
    sin_val = float(np.sin(2.0 * np.pi * phase))
    cos_val = float(np.cos(2.0 * np.pi * phase))

    # Unobserve to avoid double-rebuild
    tilt_x_slider.unobserve(rebuild_microscope, names='value')
    tilt_y_slider.unobserve(rebuild_microscope, names='value')
    if axis == 'x':
        tilt_x_slider.value = float(np.clip(amp * sin_val, _dr_lo, _dr_hi))
    elif axis == 'y':
        tilt_y_slider.value = float(np.clip(amp * sin_val, _dr_lo, _dr_hi))
    else:  # both
        tilt_x_slider.value = float(np.clip(amp * sin_val, _dr_lo, _dr_hi))
        tilt_y_slider.value = float(np.clip(amp * cos_val, _dr_lo, _dr_hi))
    tilt_x_slider.observe(rebuild_microscope, names='value')
    tilt_y_slider.observe(rebuild_microscope, names='value')

    rebuild_microscope()

    if wobble_toggle.value:
        t = threading.Timer(0.25, _wobble_tick)
        t.daemon = True
        _wobble_state['timer'] = t
        t.start()


def _on_wobble_toggle(change):
    if change['new']:
        wobble_warning.value = '⚠ Use low ray count for wobble'
        _wobble_state['phase'] = 0.0
        _wobble_tick()
    else:
        wobble_warning.value = ''
        t = _wobble_state.get('timer')
        if t is not None:
            t.cancel()
            _wobble_state['timer'] = None


# ── Solver logic ─────────────────────────────────────────────────────

def _solver_context():
    micro_now = _cache['micro']
    comps_now = list(micro_now['components'])
    labels_now = list(micro_now['labels'])
    def_idx = labels_now.index('Deflector')
    sample_idx = labels_now.index('Sample')
    def_comp = comps_now[def_idx]
    return comps_now, def_idx, sample_idx, float(def_comp.z), float(def_comp.spacing)


def _sample_state_for_solver(shift_xy, tilt_xy, shift_bal, tilt_bal):
    comps_base, def_idx, sample_idx, z_def, d_def = _solver_context()
    deflector = DoubleDeflector(
        z=z_def,
        spacing=d_def,
        shift_x=shift_xy[0],
        shift_y=shift_xy[1],
        tilt_x=tilt_xy[0],
        tilt_y=tilt_xy[1],
        shift_balance_x=shift_bal[0],
        shift_balance_y=shift_bal[1],
        tilt_balance_x=tilt_bal[0],
        tilt_balance_y=tilt_bal[1],
    )
    comps = list(comps_base)
    comps[def_idx] = deflector
    ray0 = Ray(
        x=jnp.array(0.0), y=jnp.array(0.0),
        dx=jnp.array(0.0), dy=jnp.array(0.0),
        z=jnp.array(z_source),
        pathlength=jnp.array(0.0),
    )
    out = run_to_end(ray0, comps[:sample_idx + 1])
    return jnp.stack([out.x, out.y, out.dx, out.dy])


def _solve_balance_2d(initial_phys, residual_physical):
    bounds_arr = jnp.asarray([[_bl_lo, _bl_hi], [_bl_lo, _bl_hi]], dtype=jnp.float64)
    lo, hi = bounds_arr[:, 0], bounds_arr[:, 1]
    x0_phys = jnp.clip(jnp.asarray(initial_phys, dtype=jnp.float64), lo + 1e-6, hi - 1e-6)
    s0 = jnp.clip((x0_phys - lo) / (hi - lo), 1e-9, 1.0 - 1e-9)
    u0 = jnp.log(s0) - jnp.log1p(-s0)

    def residual_unbounded(u, args):
        ba, = args
        phys = bounded_logistic_to_physical(u, ba)
        return residual_physical(phys)

    solver = optx.LevenbergMarquardt(rtol=1e-10, atol=1e-10)
    sol = optx.root_find(
        residual_unbounded,
        solver,
        y0=u0,
        args=(bounds_arr,),
        options={'jac': 'fwd'},
        max_steps=200,
        throw=False,
    )

    u_star = jnp.asarray(sol.value, dtype=jnp.float64)
    x_star = bounded_logistic_to_physical(u_star, bounds_arr)
    res = residual_physical(x_star)
    return np.asarray(x_star, dtype=float), float(jnp.linalg.norm(res)), sol


def _on_solve_shift_balance(btn):
    solve_status_label.value = 'Solving shift balance...'
    solve_shift_balance_btn.disabled = True
    solve_tilt_balance_btn.disabled = True
    try:
        tilt_bal_fixed = jnp.asarray([
            _cache['deflector_tilt_balance_x'],
            _cache['deflector_tilt_balance_y'],
        ], dtype=jnp.float64)

        def residual_shift(shift_bal_xy):
            shift_xy0 = jnp.zeros(2, dtype=jnp.float64)
            tilt_xy0 = jnp.zeros(2, dtype=jnp.float64)
            J = jax.jacfwd(
                lambda sxy: _sample_state_for_solver(sxy, tilt_xy0, shift_bal_xy, tilt_bal_fixed)
            )(shift_xy0)
            return jnp.asarray([J[2, 0], J[3, 1]], dtype=jnp.float64)

        x_star, res_norm, sol = _solve_balance_2d(
            [_cache['deflector_shift_balance_x'], _cache['deflector_shift_balance_y']],
            residual_shift,
        )
        sbx_sol, sby_sol = float(x_star[0]), float(x_star[1])

        shift_balance_x_slider.unobserve(rebuild_microscope, names='value')
        shift_balance_y_slider.unobserve(rebuild_microscope, names='value')
        shift_balance_x_slider.value = float(np.clip(sbx_sol, _bl_lo, _bl_hi))
        shift_balance_y_slider.value = float(np.clip(sby_sol, _bl_lo, _bl_hi))
        shift_balance_x_slider.observe(rebuild_microscope, names='value')
        shift_balance_y_slider.observe(rebuild_microscope, names='value')

        rebuild_microscope()

        psx = pivot_distance(DEFLECTOR_SPACING, sbx_sol)
        psy = pivot_distance(DEFLECTOR_SPACING, sby_sol)
        steps = sol.stats.get('num_steps', '?') if hasattr(sol, 'stats') else '?'
        result = getattr(sol, 'result', '?')
        solve_status_label.value = (
            f"Shift solved. bal=({sbx_sol:.4f}, {sby_sol:.4f}) "
            f"pivot=({psx*1e3:.2f} mm, {psy*1e3:.2f} mm) "
            f"|res|={res_norm:.2e} steps={steps} result={result}"
        )
    except Exception as e:
        solve_status_label.value = f"Shift solver error: {e}"
    finally:
        solve_shift_balance_btn.disabled = False
        solve_tilt_balance_btn.disabled = False


def _on_solve_tilt_balance(btn):
    solve_status_label.value = 'Solving tilt balance...'
    solve_shift_balance_btn.disabled = True
    solve_tilt_balance_btn.disabled = True
    try:
        shift_bal_fixed = jnp.asarray([
            _cache['deflector_shift_balance_x'],
            _cache['deflector_shift_balance_y'],
        ], dtype=jnp.float64)

        def residual_tilt(tilt_bal_xy):
            shift_xy0 = jnp.zeros(2, dtype=jnp.float64)
            tilt_xy0 = jnp.zeros(2, dtype=jnp.float64)
            J = jax.jacfwd(
                lambda txy: _sample_state_for_solver(shift_xy0, txy, shift_bal_fixed, tilt_bal_xy)
            )(tilt_xy0)
            return jnp.asarray([J[0, 0], J[1, 1]], dtype=jnp.float64)

        x_star, res_norm, sol = _solve_balance_2d(
            [_cache['deflector_tilt_balance_x'], _cache['deflector_tilt_balance_y']],
            residual_tilt,
        )
        tbx_sol, tby_sol = float(x_star[0]), float(x_star[1])

        tilt_balance_x_slider.unobserve(rebuild_microscope, names='value')
        tilt_balance_y_slider.unobserve(rebuild_microscope, names='value')
        tilt_balance_x_slider.value = float(np.clip(tbx_sol, _bl_lo, _bl_hi))
        tilt_balance_y_slider.value = float(np.clip(tby_sol, _bl_lo, _bl_hi))
        tilt_balance_x_slider.observe(rebuild_microscope, names='value')
        tilt_balance_y_slider.observe(rebuild_microscope, names='value')

        rebuild_microscope()

        ptx = pivot_distance(DEFLECTOR_SPACING, tbx_sol)
        pty = pivot_distance(DEFLECTOR_SPACING, tby_sol)
        steps = sol.stats.get('num_steps', '?') if hasattr(sol, 'stats') else '?'
        result = getattr(sol, 'result', '?')
        solve_status_label.value = (
            f"Tilt solved. bal=({tbx_sol:.4f}, {tby_sol:.4f}) "
            f"pivot=({ptx*1e3:.2f} mm, {pty*1e3:.2f} mm) "
            f"|res|={res_norm:.2e} steps={steps} result={result}"
        )
    except Exception as e:
        solve_status_label.value = f"Tilt solver error: {e}"
    finally:
        solve_shift_balance_btn.disabled = False
        solve_tilt_balance_btn.disabled = False


# ── Reset logic ──────────────────────────────────────────────────────

def _on_reset_deflector(btn):
    # Stop wobble
    wobble_toggle.value = False
    t = _wobble_state.get('timer')
    if t is not None:
        t.cancel()
        _wobble_state['timer'] = None

    # Unobserve all deflector sliders
    shift_x_slider.unobserve(rebuild_microscope, names='value')
    shift_y_slider.unobserve(rebuild_microscope, names='value')
    tilt_x_slider.unobserve(rebuild_microscope, names='value')
    tilt_y_slider.unobserve(rebuild_microscope, names='value')
    shift_balance_x_slider.unobserve(rebuild_microscope, names='value')
    shift_balance_y_slider.unobserve(rebuild_microscope, names='value')
    tilt_balance_x_slider.unobserve(rebuild_microscope, names='value')
    tilt_balance_y_slider.unobserve(rebuild_microscope, names='value')

    shift_x_slider.value = float(aux.get('deflector_shift_x_default', 0.0))
    shift_y_slider.value = float(aux.get('deflector_shift_y_default', 0.0))
    tilt_x_slider.value = float(aux.get('deflector_tilt_x_default', 0.0))
    tilt_y_slider.value = float(aux.get('deflector_tilt_y_default', 0.0))
    shift_balance_x_slider.value = float(aux.get('deflector_shift_balance_x_default', 1.0))
    shift_balance_y_slider.value = float(aux.get('deflector_shift_balance_y_default', 1.0))
    tilt_balance_x_slider.value = float(aux.get('deflector_tilt_balance_x_default', 1.0))
    tilt_balance_y_slider.value = float(aux.get('deflector_tilt_balance_y_default', 1.0))

    shift_x_slider.observe(rebuild_microscope, names='value')
    shift_y_slider.observe(rebuild_microscope, names='value')
    tilt_x_slider.observe(rebuild_microscope, names='value')
    tilt_y_slider.observe(rebuild_microscope, names='value')
    shift_balance_x_slider.observe(rebuild_microscope, names='value')
    shift_balance_y_slider.observe(rebuild_microscope, names='value')
    tilt_balance_x_slider.observe(rebuild_microscope, names='value')
    tilt_balance_y_slider.observe(rebuild_microscope, names='value')

    solve_status_label.value = ''
    rebuild_microscope()


# ── Wire callbacks ───────────────────────────────────────────────────

ap_x_minus.on_click(lambda _: _nudge_aperture(-1.0, 0.0))
ap_x_plus.on_click(lambda _: _nudge_aperture(+1.0, 0.0))
ap_y_minus.on_click(lambda _: _nudge_aperture(0.0, -1.0))
ap_y_plus.on_click(lambda _: _nudge_aperture(0.0, +1.0))
z_minus_btn.on_click(lambda _: _nudge_z(-1.0))
z_plus_btn.on_click(lambda _: _nudge_z(+1.0))

spot_slider.observe(rebuild_microscope, names='value')
mag_slider.observe(rebuild_microscope, names='value')
ray_count_dropdown.observe(rebuild_microscope, names='value')
aperture_dropdown.observe(rebuild_microscope, names='value')
z_slider.observe(update_beam_image, names='value')
hw_slider.observe(update_beam_image, names='value')
gain_slider.observe(update_beam_image, names='value')
auto_extent_toggle.observe(update_beam_image, names='value')
snap_dropdown.observe(on_snap, names='value')

shift_x_slider.observe(rebuild_microscope, names='value')
shift_y_slider.observe(rebuild_microscope, names='value')
tilt_x_slider.observe(rebuild_microscope, names='value')
tilt_y_slider.observe(rebuild_microscope, names='value')
shift_balance_x_slider.observe(rebuild_microscope, names='value')
shift_balance_y_slider.observe(rebuild_microscope, names='value')
tilt_balance_x_slider.observe(rebuild_microscope, names='value')
tilt_balance_y_slider.observe(rebuild_microscope, names='value')

wobble_toggle.observe(_on_wobble_toggle, names='value')
solve_shift_balance_btn.on_click(_on_solve_shift_balance)
solve_tilt_balance_btn.on_click(_on_solve_tilt_balance)
reset_deflector_btn.on_click(_on_reset_deflector)

# ── Layout ───────────────────────────────────────────────────────────

aperture_controls = widgets.VBox([
    widgets.HTML('<h4 style="margin:0 0 4px">Aperture</h4>'),
    widgets.HBox([aperture_dropdown, ap_step_dropdown]),
    widgets.HBox([ap_x_minus, ap_x_label, ap_x_plus, widgets.HTML('&nbsp;'), ap_y_minus, ap_y_label, ap_y_plus]),
], layout=widgets.Layout(border='1px solid #ddd', padding='8px', width='fit-content'))

deflector_controls = widgets.VBox([
    widgets.HTML('<h4 style="margin:0 0 4px">Double Deflector</h4>'),
    widgets.HBox([shift_x_slider, shift_y_slider]),
    widgets.HBox([tilt_x_slider, tilt_y_slider]),
    widgets.HBox([shift_balance_x_slider, shift_balance_y_slider]),
    widgets.HBox([tilt_balance_x_slider, tilt_balance_y_slider]),
    widgets.HBox([pivot_shift_x_label, pivot_shift_y_label]),
    widgets.HBox([pivot_tilt_x_label, pivot_tilt_y_label]),
    widgets.HBox([wobble_toggle, wobble_amp_slider, wobble_axis_dropdown, wobble_warning]),
    widgets.HBox([solve_shift_balance_btn, solve_tilt_balance_btn, reset_deflector_btn, solve_status_label]),
], layout=widgets.Layout(border='1px solid #ddd', padding='8px', width='fit-content'))

beam_controls = widgets.VBox([
    widgets.HTML('<h4 style="margin:8px 0 4px">Beam Slice</h4>'),
    widgets.HBox([z_slider]),
    widgets.HBox([z_minus_btn, z_plus_btn, z_step_dropdown, snap_dropdown]),
    widgets.HBox([hw_slider, gain_slider, auto_extent_toggle]),
])

controls = widgets.VBox([
    widgets.HTML('<h4 style="margin:0 0 4px">Microscope</h4>'),
    widgets.HBox([spot_slider, mag_slider, ray_count_dropdown]),
    aperture_controls,
    deflector_controls,
    beam_controls,
])


In [12]:

display(widgets.VBox([fig, controls]))


    'data': [{'colorbar': {'len': 0.6716190476190477,
                          …